# Supervisor Agent Routing

A supervisor agent acts as a dispatcher: it reads the user's query, classifies the intent, and delegates to the right specialist. The supervisor never produces the final answer itself — its only job is routing. This keeps each worker agent's prompt small and its behavior predictable.

**What you'll build:** A supervisor that routes customer support queries to one of three specialists — billing, technical support, or general FAQ — and returns the specialist's response.

**Time:** ~20 min | **Difficulty:** Intermediate

## Prerequisites & Setup

You need an `OPENAI_API_KEY` environment variable set.

**What you'll learn:**
- Configure a `SupervisorAgent` with a set of named worker agents
- Define routing rules and fallback behavior
- Inspect which worker handled a given query
- Chain the supervisor output back to the user

In [ ]:
!pip install synapsekit -q

In [ ]:
import os

# Set your API key before running
# os.environ["OPENAI_API_KEY"] = "sk-..."

## Step 1: Define Worker Agents

Each worker agent handles a specific domain: billing questions, technical issues, or general FAQ.

In [ ]:
from __future__ import annotations
import asyncio

from synapsekit.agents import Agent, SupervisorAgent
from synapsekit.llms.openai import OpenAILLM
from synapsekit import LLMConfig

llm = OpenAILLM(model="gpt-4o-mini", config=LLMConfig(temperature=0.3))

billing_agent = Agent(
    name="billing",
    instructions=(
        "You handle billing, payment, invoice, and subscription questions. "
        "Be concise and accurate. If you cannot resolve the issue, say so clearly."
    ),
    llm=llm,
)

tech_agent = Agent(
    name="tech_support",
    instructions=(
        "You handle technical issues: bugs, installation problems, API errors, "
        "and performance questions. Ask clarifying questions if needed."
    ),
    llm=llm,
)

faq_agent = Agent(
    name="faq",
    instructions=(
        "You answer general product questions, feature requests, and onboarding queries. "
        "Keep answers friendly and under 150 words."
    ),
    llm=llm,
)

## Step 2: Configure the Supervisor

The `routing_instructions` guide the supervisor's classification decision. The supervisor returns one of the keys in `workers` as its routing decision.

In [ ]:
supervisor = SupervisorAgent(
    name="support-supervisor",
    workers={
        "billing":      billing_agent,
        "tech_support": tech_agent,
        "faq":          faq_agent,
    },
    # routing_instructions guides the supervisor's classification decision.
    # The supervisor returns one of the keys in `workers` as its routing decision.
    routing_instructions=(
        "Classify the user's query into exactly one of: billing, tech_support, faq.\n"
        "- billing: questions about payments, invoices, subscriptions, refunds\n"
        "- tech_support: questions about errors, bugs, installation, API usage\n"
        "- faq: everything else — features, onboarding, general product questions\n"
        "Return only the category name, nothing else."
    ),
    llm=llm,
    # fallback defines which worker handles queries the supervisor cannot classify
    fallback="faq",
)

## Step 3: Run Queries Through the Supervisor

`result.routed_to` tells you which worker was selected — useful for logging and analytics.

In [ ]:
async def handle_query(query: str) -> str:
    result = await supervisor.arun(query)

    # result.routed_to tells you which worker was selected — useful for logging
    print(f"[supervisor] Routed to: {result.routed_to}")
    print(f"[{result.routed_to}] Response: {result.response}")

    return result.response

## Complete Working Example

Run four different queries through the supervisor — one for billing, one for technical support, and two for general FAQ.

In [ ]:
QUERIES = [
    "I was charged twice for my subscription this month.",
    "The API keeps returning a 429 error even though I'm under my rate limit.",
    "Does SynapseKit support streaming responses?",
    "How do I export my conversation history?",
]

async def main():
    for query in QUERIES:
        print(f"\nQuery: {query}")
        print("-" * 60)
        await handle_query(query)

asyncio.run(main())

## Next Steps

- **[Crew Content Pipeline](crew-content-pipeline.ipynb)** — coordinate agents with shared task context
- **[Agent Handoff Chains](handoff-chains.ipynb)** — pass accumulated context through a linear pipeline
- **[Parallel Agent Execution](parallel-agent-execution.ipynb)** — fan out to multiple agents simultaneously